# W8A8 matmul with per-channel and per-token scales

An int8 linear layer quantizes the weights once (offline) and the activations on the fly, multiplies in integers with int32 accumulation, and rescales the result back to float. One scale per tensor is too coarse, so use one scale **per output channel** for the weights and one **per token** (row) for the activations.

Write `int8_linear(x, w)` for `x` `(B, K)` and `w` `(K, N)`, both float32:

1. Weight scales, one per column $n$: $s^w_n = \max_k |w_{kn}| / 127$. Use $s^w_n = 1$ for an all-zero column.
2. Activation scales, one per row $b$: $s^x_b = \max_k |x_{bk}| / 127$, with the same zero rule.
3. Quantize: $q = \mathrm{clip}(\mathrm{round}(v / s), -127, 127)$ cast to int8, where `round` is round-half-to-even (`jnp.round`). Divide by the scale; do not multiply by its reciprocal.
4. Integer matmul: `acc = jax.lax.dot(qx, qw, preferred_element_type=jnp.int32)` gives exact int32 sums.
5. Dequantize: $y_{bn} = \text{acc}_{bn} \cdot s^x_b \cdot s^w_n$.

Return `(y, acc, qw, sw)`: `y` `(B, N)` float32, `acc` `(B, N)` int32, `qw` `(K, N)` int8 and `sw` `(N,)` float32.

This problem runs in **Colab** (JAX isn't available in the browser). Use the default CPU runtime; JAX computes in float32 here. The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import jax
import jax.numpy as jnp

def int8_linear(x, w):
    """Return (y float32 (B, N), acc int32 (B, N), qw int8 (K, N), sw float32 (N,))."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[1.0, -1.0]], "dtype": "float64"}, {"__nd__": [[1.0, 0.5], [2.0, -0.5]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[-0.9921259880065918, 1.0]], "dtype": "float32"}, {"__nd__": [[-8001, 32258]], "dtype": "int32"}, {"__nd__": [[64, 127], [127, -127]], "dtype": "int8"}, {"__nd__": [0.015748031437397003, 0.003937007859349251], "dtype": "float32"}]}}, {"a": [{"__nd__": [[0.0, 0.0], [127.0, 63.5]], "dtype": "float64"}, {"__nd__": [[1.0, 0.0], [0.0, 0.0]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[0.0, 0.0], [127.0, 0.0]], "dtype": "float32"}, {"__nd__": [[0, 0], [16129, 0]], "dtype": "int32"}, {"__nd__": [[127, 0], [0, 0]], "dtype": "int8"}, {"__nd__": [0.007874015718698502, 1.0], "dtype": "float32"}]}}, {"a": [{"__nd__": [[1.1827683822490698, 0.9800081476549486, 2.2666932138811453, -0.14220093149203353, -0.5799002829800761, -1.4470425027291827, -3.859277514716476, -1.308398340087321], [0.48681855045100436, -0.8855311413697625, -2.5948239990829336, 0.6082597078488055, -0.5916575081322901, -0.36100409352785295, -0.43143950165476136, -0.13469839560367824], [0.47444407695595, -1.7345166510884678, -0.22711744271967574, -0.5085891107173511, 1.1367247203334006, -1.0352973533071197, 1.6374114454309625, 0.6080331975223119]], "dtype": "float64"}, {"__nd__": [[1.6246433863782133, -0.7044853751017233, 0.8059521152228815, 0.3631040001715692, -1.497670353579111], [-0.26091789722820546, -0.4526963321434531, -0.9661537662976033, -0.7681495562031, -1.4398268099452578], [-0.4671738791781496, 0.3341363029795037, -0.7788386330876441, -1.5631519863854781, 1.0453300853245673], [-0.37081461226831575, 0.027424727629300243, -0.23394341341574568, -0.5401185262342673, 0.6017587397889111], [-0.5299062148166881, 0.028727294300772213, -1.4449418453788945, -1.6387657921658814, -0.38955695537009144], [1.9106715035731066, 0.1812625943452766, 1.9336300601288359, 0.4521845499496128, 0.4534593078626402], [-0.5833236968021522, -0.022417566518173702, 1.7912857198271908, 1.1851895804822443, -0.6261892903522636], [-0.46981695997984013, 1.337241742949975, -0.5147852634626356, 0.9197404973376132, -0.9095622374384135]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[1.0620226860046387, -2.4586451053619385, -9.953895568847656, -9.291991233825684, 2.2963459491729736], [1.9288579225540161, -1.0698646306991577, 2.5623764991760254, 4.741104602813721, -1.3613355159759521], [-2.3097341060638428, 0.991263747215271, 1.3248052597045898, 2.334209680557251, -1.2567498683929443]], "dtype": "float32"}, {"__nd__": [[2323, -7684, -21514, -23697, 6408], [6275, -4973, 8237, 17983, -5650], [-11241, 6893, 6371, 13245, -7803]], "dtype": "int32"}, {"__nd__": [[108, -67, 53, 28, -127], [-17, -43, -63, -60, -122], [-31, 32, -51, -121, 89], [-25, 3, -15, -42, 51], [-35, 3, -95, -127, -33], [127, 17, 127, 35, 38], [-39, -2, 118, 92, -53], [-31, 127, -34, 71, -77]], "dtype": "int8"}, {"__nd__": [0.01504465751349926, 0.01052946224808693, 0.015225433744490147, 0.012903667986392975, 0.011792680248618126], "dtype": "float32"}]}}, {"a": [{"__nd__": [[0.6280165940626083, 1.5262020846668078, 0.835006130784672, 0.24344399384005985, 0.14039998297355255, -0.08455620551134427, -0.023061339558410495, 1.1788224355661225, 1.15390491426208, 0.40781216959604366, -1.2592136077645948, -0.520658641532731, 0.29740088482682187, 0.05149551908664256, -1.3529665793595937, 0.6640093321096796], [2.5988309831504903, -1.2265409197470718, 0.06377375209811639, 1.61978856621042, -0.41734810360675173, -0.40774174906919597, 0.16957523098743163, 0.5669816138670667, -0.4111957110197731, 0.9798143542823354, -1.9049970217947472, 0.655421857244873, 0.36877126586848974, -0.15890898812285262, 0.38736743152539377, 0.04787779318377918], [1.329818670087016, -0.4617116581712759, -0.04141893043933976, -1.387654881387399, -1.1247761756336008, -0.24098292687567094, -1.5560283497926257, -0.8472880919393981, -1.3260211766736796, -0.493601948194795, 0.11743839429605914, 0.9435369892862817, -1.9406078816289432, -0.20033897046056426, 0.8657073055485704, 1.3801302841729033], [1.4060709502619853, -1.4343265844965274, -0.5131397260884635, 0.5641874893843708, 0.9387234898768438, 0.2820188282897474, 0.26558640328963834, 0.24109857143139354, -0.767360786337068, 0.16129042571022478, 0.812236820154686, -0.33907038376846577, 1.1728353212200668, -2.8421345556851825, -0.32651169373316946, 1.700220235506148], [0.8735536573431392, 1.4441728528739437, 0.14612519800614662, -1.1817241461859231, -0.5104510153909563, 1.52219718663443, -1.8365815440219238, -0.9536317245125848, -1.7139344118290056, -0.5336349645185658, -0.4212379900574514, 0.5799363404994193, -0.823915421428668, 1.7135000303244519, 0.6477273898405688, -0.47090516987550435]], "dtype": "float64"}, {"__nd__": [[0.5532605888887387, 10.880030460973694, -0.057989987866526184, -2.3189360947215656], [0.4314941677088505, -106.3139892225441, 0.9099212170226342, 0.6059655730064136], [0.8300566485784159, 41.38491718576939, 0.2985144698332214, -0.5350014137339273], [-0.3070622870057959, 75.40361357860671, -0.5822293778059019, -0.22812380385128453], [-0.7245145044162855, -25.86246687453911, -0.30655788195456973, 0.2565516458675206], [-0.2938560550257995, -17.73542324943842, -0.6168151525371501, 0.08958446690608873], [-1.3441660008066356, 2.609244965388834, 1.3100257323485311, -0.7669128788338543], [-0.04539173803348671, 140.86045082421214, -0.9926744097609564, -1.621141524211868], [-0.42680913157992273, 90.34689901890843, 2.035115511784091, -1.2159872631305197], [0.5701405884703373, 0.22607156715674592, -0.5883707554353481, -1.9456978173547614], [1.0245839797071592, -43.02691533650394, 0.29900768417420936, 0.08189396317057414], [1.8701812688972848, 58.799739470679356, -1.1553360632608776, -0.2623951558776075], [1.0368475527289776, -66.65074821406695, -0.3922883318669356, 0.22757580070206257], [-1.6158261586823717, -61.772907321839334, -1.7040770041170938, 0.28697587878613073], [-0.5615144689797945, 46.43241438227477, 0.11379425356523251, 0.5720172533764897], [-0.08367584391494008, 37.59618976228316, -0.10331715721652544, -0.9822015508199493]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[-0.07998835295438766, 126.77098083496094, 2.1779918670654297, -6.39766263961792], [1.077284336090088, 472.1334533691406, -4.756389617919922, -9.863859176635742], [3.8511545658111572, 33.0317268371582, -2.857771396636963, 0.22181262075901031], [4.9036993980407715, 219.99205017089844, 0.9370562434196472, -5.994542598724365], [1.151656985282898, -521.6859741210938, -6.684863090515137, 6.140295028686523]], "dtype": "float32"}, {"__nd__": [[-452, 9511, 11310, -29156], [3575, 20802, -14505, -26399], [17115, 1949, -11671, 795], [14880, 8863, 2613, -14670], [5408, -32525, -28847, 23254]], "dtype": "int32"}, {"__nd__": [[38, 10, -4, -127], [29, -96, 57, 33], [56, 37, 19, -29], [-21, 68, -36, -12], [-49, -23, -19, 14], [-20, -16, -38, 5], [-91, 2, 82, -42], [-3, 127, -62, -89], [-29, 81, 127, -67], [39, 0, -37, -107], [70, -39, 19, 4], [127, 53, -72, -14], [70, -60, -24, 12], [-110, -56, -106, 16], [-38, 42, 7, 31], [-6, 34, -6, -54]], "dtype": "int8"}, {"__nd__": [0.014725836925208569, 1.1091374158859253, 0.016024531796574593, 0.018259339034557343], "dtype": "float32"}]}}, {"a": [{"__nd__": [[-0.9910645338301242, -1.4978566295734526, 0.12336137414890104, 1.983714638212815, -0.16046464512974998, -1.0261433361766163, -0.05588164653826363, -0.5139143393319071, -0.901469713862993, -1.893521073060943, -0.053416097268674, 1.4888947585205548, 1.2314825317133544, 0.9875729460028575, 0.21489765417894124, 0.47746618437907484, 1.0171842958596853, -0.047759930102910526, 0.2893196184637676, -0.3914239053578755, 1.140783208584205, -0.9182083587191213, 0.6901000340614477, 0.8650198290418158, 0.8460553055003698, 0.7213915093385939, -0.4541162569906305, -0.5537050377425773, -0.40141320828001226, -0.3641275684342096, -1.1150614694493721, -0.4344430888960559], [-62.53801681907678, -93.68719848731347, 37.56749135768645, -74.4254517322277, 26.249937333996293, 148.677631252472, 170.28700586993813, 3.404252264278468, 103.81714995744854, 26.66570820287929, 241.0959609000124, 166.55972856414405, 113.67963412352344, -13.834614130453993, -23.239173752978797, 56.63440245740382, -1.3940134104685085, -22.86840592004568, 71.56772374312473, -123.35605384426214, -198.92985863636608, -100.32833436493345, -114.47900189737565, -6.800987354659408, -85.92685641415429, -18.273095943168407, -201.759274191572, 148.99283222018548, 41.06974578461337, -85.54072499021402, 32.953840860265096, 124.8591231184819]], "dtype": "float64"}, {"__nd__": [[0.8575121266794775, -0.07836388230610881, 0.6296101058342439, 0.901539067592392, -1.2027824036812047, -1.4354318442447123], [-0.5858865853099267, 1.0009539605743045, 1.4963227011742488, 0.14631363979616674, 1.5255658786331128, 1.016129281776467], [-0.059700986218757565, -0.7373328349120187, -0.6347468957174698, 0.09383003523679159, 1.4256720488464023, 0.8571510352635652], [-1.3003236775472613, -1.2231372636159012, 0.8309051921212542, 0.21027028626919175, -0.9396619893465503, 0.6950614168575278], [1.3884080538511916, 0.8171475302908839, 0.12215804726956613, 1.3605958857826965, -0.3735580691036685, -0.5804498795997574], [-0.9129789298634137, 1.9217032637244162, 2.1532999663605445, -0.024281463734012982, 0.6140634482474299, 0.47401865862733406], [0.9607584696141868, 1.2197283900354003, 0.705481979187263, -0.07205161473307659, -0.5003237058261072, -1.0049263685016512], [-2.6094582134903725, -0.07576550886977991, -0.817176625060308, 0.05142107609694558, -0.0847632381316415, -1.012488428139888], [-0.48338356105366587, 0.43655552728884944, 0.18971776597643028, -0.8664518380813803, -0.7378674254288081, 0.5928539404329747], [0.2585746116795385, 0.32925735550295243, -0.144748560010324, 0.7629107587422029, 0.11412035721971718, 2.557112209761941], [-0.454526610983739, -0.6871369880154431, 1.6766412459746578, -0.05408543460530651, -0.7640969066116208, 0.45277685963891395], [-2.3062521239791374, 0.4649412005017543, 0.03599127651611266, 0.2849421342796853, -0.7903070223846702, 1.3343582938702274], [0.33078820660006625, -0.004089997857936735, 0.37822772435906477, -0.8227179138399797, -0.11454635400374913, -1.4914438634192537], [-1.0933658157048933, 0.3066609749994623, 0.12954327783472816, 0.7543774163599632, -0.6539197113735622, -0.10211207577058908], [-0.6794628696491911, 0.26395801331764274, 0.7900763964031752, 1.075993048013171, 0.5764504671662501, -0.7898589394360896], [0.21316320919385914, -1.0364885010666045, -1.9214488639194578, 0.17315780312533796, 0.08208643329512258, 0.7411002135780272], [0.8213272660461594, -1.2388105107914116, -0.9146661842117264, -0.01304258968200663, 0.0636571576750391, 0.39916838642744834], [-1.3178111882476198, -0.26341896603411485, 0.6289007047356723, 1.8828318305090208, -1.2688267061761878, 1.605264421945028], [-0.3019381696531152, -1.4603236761927478, 1.3134790017289018, -1.0860508363919765, -1.187252811442023, 0.6983133467975097], [1.0149434288257406, 0.04542681511723151, -0.7440102448727568, -1.5148690288413447, 0.08267143613979593, -2.3637042226219442], [-1.482310835296758, 1.0308399930068024, 0.5897674349683752, 1.189474706490783, 0.37670143252552796, -1.1753494116182133], [0.43693093305720426, -1.8453704739528716, 0.6581871650214526, 1.3261683934876929, -1.0405556845107033, 0.986209741497253], [-0.0029098795250363098, -0.20389078034935193, -2.0172003657999125, -0.20042432191697002, 0.8230373717303825, -1.7599184629959224], [-0.5283723573590631, -0.7507297210471999, -1.6745860195820719, 0.997359557033254, -0.02731258298736381, -0.004999852401579111], [0.9364438543837886, -0.7983181945814336, 3.2085921360327316, -0.5053469144762512, -1.2078464166591396, 0.5913527431653398], [0.34000365905220237, -0.9962452593378184, -0.9367339198202601, 0.8025326480832135, 0.26687076455047115, 0.39078380275494706], [0.8495445985078055, -0.621765797295769, 1.246840633654848, 0.01709578817885511, 0.7108683973878703, -0.9244765514585456], [0.7468294777927093, 0.27711759969086525, 2.4762073323741025, -0.15000838401427574, 1.532205838061597, 0.6430677931202534], [1.5768531664420402, 0.24867335439343105, 0.49467080845503375, -2.201864880558068, -1.2614710169282073, 1.0599310868487861], [0.43970311822212305, 0.7889237041384558, -0.24403664416520585, 0.8018409941574018, -0.3383093041669525, -0.038959626361131545], [1.2324993246268356, 1.0950714464789963, -0.41731182913013487, 0.06914458727381313, 0.2545087998091163, -0.12289066149392434], [-0.08211074196462763, -0.9231190193770936, 0.5842838306753202, -1.025925409448804, -0.9446524199043005, 1.2855982541642956]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[-8.697759628295898, -8.595388412475586, -6.059884548187256, 1.1682682037353516, -4.132638931274414, -5.5773539543151855], [-256.73834228515625, 410.7757873535156, 709.0697631835938, -789.231201171875, -430.58355712890625, 1273.803466796875]], "dtype": "float32"}, {"__nd__": [[-27101, -36367, -15356, 4314, -21930, -17734], [-6582, 14300, 14784, -23979, -18800, 33325]], "dtype": "int32"}, {"__nd__": [[42, -5, 25, 52, -100, -71], [-29, 66, 59, 8, 126, 50], [-3, -49, -25, 5, 118, 43], [-63, -81, 33, 12, -78, 35], [68, 54, 5, 78, -31, -29], [-44, 127, 85, -1, 51, 24], [47, 81, 28, -4, -41, -50], [-127, -5, -32, 3, -7, -50], [-24, 29, 8, -50, -61, 29], [13, 22, -6, 44, 9, 127], [-22, -45, 66, -3, -63, 22], [-112, 31, 1, 16, -66, 66], [16, 0, 15, -47, -9, -74], [-53, 20, 5, 44, -54, -5], [-33, 17, 31, 62, 48, -39], [10, -68, -76, 10, 7, 37], [40, -82, -36, -1, 5, 20], [-64, -17, 25, 109, -105, 80], [-15, -97, 52, -63, -98, 35], [49, 3, -29, -87, 7, -117], [-72, 68, 23, 69, 31, -58], [21, -122, 26, 76, -86, 49], [0, -13, -80, -12, 68, -87], [-26, -50, -66, 58, -2, 0], [46, -53, 127, -29, -100, 29], [17, -66, -37, 46, 22, 19], [41, -41, 49, 1, 59, -46], [36, 18, 98, -9, 127, 32], [77, 16, 20, -127, -105, 53], [21, 52, -10, 46, -28, -2], [60, 72, -17, 4, 21, -6], [-4, -61, 23, -59, -78, 64]], "dtype": "int8"}, {"__nd__": [0.020546915009617805, 0.015131521038711071, 0.025264505296945572, 0.01733751967549324, 0.012064612470567226, 0.020134741440415382], "dtype": "float32"}]}}, {"a": [{"__nd__": [[0.001841137467508027, 0.0018604995331973845, 0.0003696952948661359, -0.0005179529250903714, 0.0007576761265691543, 0.0015643556597096156, 0.00013373735961503958, -0.00038578683935887206, -0.0016468723347681416, -0.0016673224859805875, 0.001730100821469481, -0.0014496460890163917, -0.0006526075091639889, -0.00028534686890047454, 0.00025652288775725065, -0.0012813078032029254, 0.0005781162830596163, 0.0005850683151943037, 7.942601781027769e-05, -0.0006128587947985613, -0.0003635603096462718, -0.001918597261337235, 0.0006283896423291539, -0.0005201870127753378, -0.0013649257096743264, -3.841585318158564e-05, 0.0004920761233583406, 0.0003522579951053345, 0.0006164907652678919, 0.0011578606461363752, -0.0005324249494756302, 0.00017826716216959418, 0.00024272105645334116, -0.001465654038257072, 0.0010709448505413117, -0.0003445902161414467, 0.00022703240139228834, 0.000277419691803443, -0.0008351902140357224, 0.0007946356730459062, 0.0009468683307119643, 0.00014567473367923625, -0.0029114946192595925, 0.0013329294835233312, -0.00017309669088168652, -0.0006708753179972297, -0.0006093140073231183, -0.001106786685719267, -0.00039451513222260343, 0.0005884045584693253, -0.0011306200493132904, 0.00018222909457308394, -0.000621371781269219, -0.0002855627454653144, -0.0003966245633714806, 0.00027789147144878355, 0.0009811198281843142, -3.923555091974866e-05, 0.0008676729342740999, 0.0014553311754124034, -0.0014719917997375737, 0.0007554655604793512, -0.0003248635007396217, -0.00021025939578572846]], "dtype": "float64"}, {"__nd__": [[0.0009468619278266308, -0.00025666967115537564, -0.0006499867092805569], [0.0012402544554855574, 0.0010293470088736255, 0.0003991409377279457], [0.002805813244700589, -0.000992356404472947, -0.0016288311697665596], [-0.00021961920196642993, -0.0011381818145785425, 0.00036606762129655254], [-0.0005910554823274746, -0.000599929286653096, 6.731384154563516e-06], [-0.0008408990863393093, -0.0019172867135949922, -0.0007267630642482367], [0.0007000660753285552, 0.00023404650578689825, 0.0009294315908025059], [0.0016780089094257083, 0.0008752078085436523, -5.879754155944812e-05], [-0.0007495385471686026, 0.00046135795266939516, 0.0016014427528698231], [0.0014188722473912642, 0.0009328629236902818, -0.0008006201469176934], [0.0005810420307493824, 0.0007062654297961499, 0.001223223107041293], [-0.0012633093172353847, -0.0006939934116076121, 0.00013807909570420003], [0.00012484216392454831, -0.0006069092765812635, 0.0004997208257260341], [-0.0002927541489449597, 0.0011051142347629395, -0.0004851699780914362], [0.00019844459511866253, -0.0020906994486125947, 0.0002031612052744749], [-0.0006621917978198481, 0.0007080321206282764, -0.00017094871473373702], [0.0012266935499985869, -0.0005294685382890349, -0.0014316061840807276], [0.0002394268642869366, 0.0005795039549335359, -0.0007422054675024669], [-0.0012922992645986542, 0.0007551930882205851, 0.001050581245866634], [0.0007304855056610187, 9.088828726391032e-05, 1.2510552822948322e-06], [0.00162978654447218, 0.000801575260948583, -0.0009025446372968787], [-0.0010707812369506258, 0.0006260390559739241, 0.0007324926339335227], [-0.0006077111829802501, -0.0007525943498335237, 0.0009662163974674023], [-0.000310275854184961, 0.0008371483380315173, -0.001097266419876259], [-0.0009802282874682008, -0.0017000600308796244, -2.699499480402971e-05], [-0.0003967001864849173, -0.001669042426724841, 0.0001587799886973623], [-0.0005231709873577695, -0.00021761491970818313, 0.001397121848646859], [0.0009693170517899208, 0.0004947219527496489, -0.0003268887450232853], [-0.0011829593412708607, -0.0006518041514510096, -0.00037179041953977536], [0.0012549564494255731, -0.000920362623187538, -0.000541000063342038], [-8.453300836832583e-06, 0.0003904193981964983, 0.0010798059128397338], [0.0013245890570392843, -0.00014784350979457668, 0.001921690385981471], [0.0006709038062725346, -0.0007827560615183228, -0.0009739353207194393], [-0.0004172091409061384, -0.0002516007701798054, 0.0007862079124482242], [-0.0008154555337854503, -0.0005698144659771811, -0.0008488834315940029], [0.0002645783332290238, -0.0014512047054396702, 0.0005469889839077605], [-0.000791116044309921, -0.00021879202556888717, 0.0004285134221720177], [0.00029454676487041225, 0.0027727696365574084, -0.0009097311747993936], [-0.0020825614454486187, -0.0013105173105305461, 0.0008228649757346172], [-0.0010701340613655122, 0.0010204728511595466, -0.00022047232514340662], [0.0007376318407059982, -0.0015292674740312534, 0.001034396602867065], [0.0011504166295700938, -0.00093286013954058, 0.001313292911113175], [0.0026240936712711212, -4.135966048570642e-06, -0.0009372209099945015], [-0.000598829367754243, -0.00028976783162202815, 0.0010054833118715318], [0.001823472544467632, -0.0004171739596086714, -0.00022682803233960746], [-0.0013859578922969943, -0.0005233338488864525, 0.0004689403292233161], [-0.0020052843312042537, 0.000686361240060903, 0.0008921758810329525], [0.0012033289314019604, -0.0005711106994030252, -0.0015662008100863847], [-0.00027948737831500305, -0.0010345554769262562, -0.0017601386559716618], [-0.0007605009678707935, -0.0013914966829627576, 0.0019210197829883841], [0.0007446890049878888, 0.0008771436547631646, -0.0011091452748537465], [0.001368516193016706, -0.0003756922445911831, 0.0008487787806965048], [-0.001019092098782545, 0.0006221648235433628, -0.0016738812363481191], [-0.0001743073315111774, -0.0002436652892419362, -0.001365821049278069], [2.5402776725066337e-05, -0.0005693309091497338, -0.0003962865296188765], [0.0007347125666937374, 0.0012915824000341689, -0.00019941350634585734], [0.0012803326500782659, 0.000998803591448924, 0.0007428821456001477], [0.0015842301605661735, -0.0015021615384671135, 0.0013277346237143815], [0.0003766833579274061, -0.0003896592697267713, -0.00016670907639975262], [0.0010168512597473057, 0.0012004227030771066, 0.0007184134848273589], [-0.0011536706785020825, -0.001454962221111557, 4.9825207210365475e-05], [0.0006409979587402515, -0.0006808535466279454, 0.0008879896494075356], [0.0006180847375866909, 0.0004930024522555049, -0.0007488386513580241], [0.001253019135967894, 0.0009798921340856798, 0.0013663952241979146]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[7.517262929468416e-06, -1.254806193173863e-06, 6.338723778753774e-06]], "dtype": "float32"}, {"__nd__": [[14842, -2507, 18273]], "dtype": "int32"}, {"__nd__": [[43, -12, -43], [56, 47, 26], [127, -45, -108], [-10, -52, 24], [-27, -27, 0], [-38, -88, -48], [32, 11, 61], [76, 40, -4], [-34, 21, 106], [64, 43, -53], [26, 32, 81], [-57, -32, 9], [6, -28, 33], [-13, 51, -32], [9, -96, 13], [-30, 32, -11], [56, -24, -95], [11, 27, -49], [-58, 35, 69], [33, 4, 0], [74, 37, -60], [-48, 29, 48], [-28, -34, 64], [-14, 38, -73], [-44, -78, -2], [-18, -76, 10], [-24, -10, 92], [44, 23, -22], [-54, -30, -25], [57, -42, -36], [0, 18, 71], [60, -7, 127], [30, -36, -64], [-19, -12, 52], [-37, -26, -56], [12, -66, 36], [-36, -10, 28], [13, 127, -60], [-94, -60, 54], [-48, 47, -15], [33, -70, 68], [52, -43, 87], [119, 0, -62], [-27, -13, 66], [83, -19, -15], [-63, -24, 31], [-91, 31, 59], [54, -26, -104], [-13, -47, -116], [-34, -64, 127], [34, 40, -73], [62, -17, 56], [-46, 28, -111], [-8, -11, -90], [1, -26, -26], [33, 59, -13], [58, 46, 49], [72, -69, 88], [17, -18, -11], [46, 55, 47], [-52, -67, 3], [29, -31, 59], [28, 23, -49], [57, 45, 90]], "dtype": "int8"}, {"__nd__": [2.209301783295814e-05, 2.1832831407664344e-05, 1.5131420695979614e-05], "dtype": "float32"}]}}, {"a": [{"__nd__": [[0.3931637177099155, -1.2456696295480638, 0.39557178225192124, 0.8807523603104813, -0.12266131212771957, -1.1239578575352644, 0.5728254565977435, 0.6560747978290971, -1.234647812827428, -0.7099828833327797, -1.4313474654001381, 0.33632506737766144, 0.18755843887642307, 1.74112558241004, -0.9056246561118265, 1.332700409994744, -0.21326582558697313, 0.2731374130446113, -1.2279609400576754, 0.02745634208899877, -0.5256289459574373, -0.10867234312869549, 0.1852686886475968, 0.4858737025992133, 0.8332617556486803, -0.01901500418432362, -0.8876161781862593, 1.0252141667088392, 0.08445314052507572, -0.5998807000093361, 1.3708963922190966, 0.2926646941519549, 0.7268860001099108, -1.6833578874884314, -0.6154655920770766, 2.0145754830437275, 0.5064280391692965, 0.48757330288958706, -0.7003826725856866, -0.13177736333659115, 0.37353626837033593, -0.07055231986129175, -0.10935651299091159, -1.7477364004600726, -0.7937045727505502, -0.4497174785747778, 0.5612155306214774, 0.5131442279379855, -0.016888872179746224, -0.7268568539469007, -0.7277840796309549, 0.23913126227050152, -1.3565813162576563, -0.2299559180867582, -0.32323918880682784, -0.26601106425856696, 1.6385977357040302, -0.15842331624971137, 0.843079018354046, 1.5733319897537008, 0.4969838779744485, -0.013954087511212073, -1.2648763176691153, 1.5974521381906124, -1.122349968451115, -2.0527278493521295, 1.040544876996442, -1.4820838073169327, -1.2184689249078464, -0.7392051504287463, -1.2139778708652353, -2.2058882545698024, 0.15221102297700553, 0.44060174164202154, 0.7101985944997663, -0.0755428320560451, -0.4461118686537844, -0.5780855290286251, 1.161352085945062, 0.7869667745152893, -0.4898326203225064, 0.253725918041413, -1.2298144765833579, 0.4437763767819972, -1.125881917443417, -0.20303715393130656, -0.29170230707004485, -2.72949476411703, 0.6065183038234604, -0.09155299588478893, -0.8432993391392263, -0.7811213497656956, -0.17083879074001793, -0.23450903722973013, 0.3747292270906053, 0.0752562877137032, 0.5309004553101253, 0.1701821939630019, -1.4566030020031626, -1.463392882335424, 0.39586752564352856, 0.32212068122844995, -0.07175849647382919, -1.3636059404479224, -0.7652481282772898, -0.8680029914389982, 0.5961969314510868, -0.8629761453986527, -1.406500642806449, -0.8508529298592826, 0.24457952860695792, 1.5060283566082913, -0.5493628657297736, 1.0526273536800648, -0.4994969223655443, -0.48220324146381477, -0.8968253104604853, 0.731034747398911, -0.5298676178831467, 0.8892872246328515, -0.44590893617062693, 1.6815957495225828, 0.17787153872087164, 0.01756606149203206, -1.2119886950445729, 1.247655033271242, 0.2720725521198472, 0.05300189823099132, 0.001210243361143907, -0.5079870598413175, -0.5295210979659518, 0.32802704927252485, -0.8045902849134438, -0.15074320563921842, 2.2316359296654174, -0.13227061730085593, -0.28537840772884593, 0.8857454183637029, -0.9709097165115476, 1.3821670780259987, 0.004175652633749179, -1.1557309760748478, -0.14295946228129702, -0.6880661031651838, 0.6843796032665412, -0.45940224530748364, 0.7135746382653129, -1.906030379533845, -0.4347338948909908, 0.561227446158101, 0.49888133581501065, 0.05973702397273116, 0.7417016268610525, 0.7699157730179412, -0.0032576710880691277, 0.20822442666581598, 0.8895421533716071, 0.16405344129826083, -2.5390843888843095, -1.5636194582967544, -0.6504500459404837, 0.675501175607009, 0.06208863477565752, 0.6394051006896142, 0.9926197042821899, 0.5355216951108854, -0.939752239565339, 0.8263573887574565, -1.3464643515101364, -1.2847316689622443, -0.45461751897892944, -2.273414290616197, 1.6290410716405979, 2.311420699437719, -0.20568195383209892, -0.42091872854357704, -2.1203127706800995, 0.4719715375225549, -0.7520889903537309, -1.2596086673008353, -1.443057112620326, -0.19124640611016874, -0.29116333455827664, -0.20778397599446213, -0.6913910921095268, -1.7973066559872866, 1.3384639708849206, 0.7213660823498207, -0.6715665747305314, -1.4313443327480653, -1.091639183139989, 0.5965275258422021, 0.41433797328286853, 0.19206860944785084, 1.097497625118322, -0.6362214127766032, 1.3339218635900538, 0.0810142389631069, -0.8314929866187135, 0.42402786026286593, 0.2349181459829905, -0.7774540783263683, 1.6982637857552185, -1.711996767522461, 0.17569234466994996, 0.3266561247223036, -0.785405785609957, -0.02360185202614001, 1.2283653333855367, -1.1827798073180562, -1.1473008932145756, -0.40620204721971276, 0.4368177383535663, 1.046201594261254, 0.5126277682938065, -1.1783499188360742, 1.0426621545897732, 0.6451654209846609, 0.957493284074515, -1.7130760873761885, -1.7566636502099935, -0.7539119477898111, -0.5382565854677068, -0.10744168627682889, -1.1883167558852636, -1.1661905998680644, 0.3178266419450796, -2.2187528532236684, -1.286144656369085, -0.08845350825577965, 0.6307635152200086, 0.16007870278128802, 1.471423929191111, 0.18842589203493113, 0.5896370075214318, 0.694341142287697, -0.07285511373675503, -1.4250648386302953, -0.4012161590606012, -0.32893723144377207, -0.9915764551741502, 0.5325554673844378, -0.6605821090903279, -1.6167686292684362, -0.15243226462560236, -1.1447778393180683, 0.3797584879820209, -2.7471965102087865, -1.8370218532304494, -0.14150245570911665, 0.09818787085466599, -0.06975359199520662, 1.3555197922318682, -0.3959174612745289, -0.6593889172755989, 0.6747562690112419], [-0.47089110368835163, 0.24381741185398662, 1.1066944293393046, -0.3048211295987791, -0.2502671099918833, -0.7442414206263429, 0.2507014409519231, 0.48155863453716036, 0.8964364090586482, 0.5348149816891989, 1.8587081033718353, 0.03881190218599468, 0.019234740118215736, -0.3872338036163459, 1.1618333997180228, -0.016379527556968802, -0.3785937424185882, -1.379839297351497, -0.21733562739766626, 0.7669184285923643, -2.6399268903237063, -1.1815387040467096, -0.9871428562579371, -0.8885277388509264, -0.03223759334474776, -0.45277377310143463, 0.058493702812507334, 0.21339387138646113, 1.2663687368721919, -0.3152801065069317, 0.2769175788655929, -0.2851698447455854, 0.0986006026643236, -0.39804856556222307, -1.1937756685093381, 0.33460276040086356, 1.0161936167311914, -0.9801459289947184, -0.08358979579036488, -0.08478452982678401, -2.146649391644349, 1.8612860463064083, 1.3905996771080436, 1.1117411434344078, 0.930913261610238, -0.8948503685969081, -2.4665972882485043, -0.11337461943554085, 0.029910594749998007, -0.3195662652970119, -0.48717547834519614, 0.9205191479147827, 0.7145887852607627, -0.07401518774332363, -0.5506893528091285, 2.1912715982675244, 0.7631148096637171, 0.04840232772664253, -1.3405809262232506, -1.5820209133568046, -2.785242516328747, -0.3051821694181968, 0.23238442542128662, -0.23007951023633538, -0.05739745598130901, 0.18503074620347149, 0.7710484824547805, -1.8070224112908284, -1.2853379303856092, 0.8231209762642537, 0.2224345456004146, 0.4424167355942122, -1.043820181624591, 1.438979213902493, 0.0680469722837637, -0.2781456930212961, -0.9200713197461512, 0.1437273057250785, 0.12197061433668571, -0.15579576406769818, 1.5637145820990275, 1.1406740134043032, -0.04282221758605302, -0.6037801423309522, -0.46430222051523395, 0.3115899234504225, -2.7755685472031884, 0.2831270132497432, -0.23776929916450523, 2.091703415022348, 0.4227685623341897, -0.17701807528535335, 0.042862401748396624, -0.6293270378738115, -0.6922299540292488, -0.06498717364947536, -1.9851882577646474, 0.006180569064946582, -1.6607671095567142, -0.7083161220611683, -0.7209028336078417, 0.6941075561045023, 1.5543119887336987, 1.417165862142472, -0.472981392229349, -0.22133099011332868, 0.044146857823847585, 0.16306448536035903, 1.1211846092001276, 0.6806917552803403, 0.004824339662849853, 1.9223918481630278, 0.9079233775972183, 1.8408974385131567, -0.0383025534538785, 0.5924000207267175, 0.3535457328887597, 0.20152739992767893, -1.0104086071545373, 0.2792496721981665, -1.4179018538870496, 0.41256024806192576, 1.741800022935734, 1.162574199743097, 0.33100216219935685, -0.2532220132794782, 0.793781650069822, -0.9992871947548548, -1.1647315181203834, -1.2602839510933397, 0.41939527701609947, -1.2780047096470812, 0.6163689235312296, 1.0386025397357197, 0.7079592400677711, -0.07626348431967657, 0.44679387854566077, -1.1630830498319094, 0.2247697653852665, 1.6885390841221832, 1.4738171710001056, -1.5462728136874906, -0.049095102030464174, -0.3126593747095671, 0.9246054873763656, 0.28908212926886523, -0.0799034160665519, -0.6176174674428488, 0.16869342102991708, -0.12718319075968007, 0.07517359736951434, -0.1354805158254263, -1.2086438242775102, 0.419598666813655, 0.4602831977823349, 0.22944195080054816, 0.26925621644203296, 1.2208171320114705, 1.0906021190969277, 1.6561396193781697, 0.7037339380399067, 0.854314279069522, 0.06975698088703851, -1.0153375286026138, -1.1435259536928482, 0.5302061578482534, 0.14208858305282673, 0.5563811615641678, -2.274070971559223, -0.21234388348695407, 0.10499183880227375, -0.3837833365144292, -0.28482592069502505, -1.8100282068050164, -0.1452114688562218, 1.2107146593855478, -1.1757011099558812, 0.4971203228663326, -0.7651481462060713, 0.1285091854471717, 1.366373832274568, 0.007231497410359335, 0.3229543324687595, 0.5954617039559593, -0.07246008548773011, 1.2063919509999868, -2.0251053336813163, -0.35679691619599607, -1.9321661575798998, -1.5477422269474275, -0.05006840300382485, -1.239146050877804, -0.44291331143645873, 1.3050181419660587, 0.5028702662358115, 3.692210082856903, -0.13569942914797492, 0.03628997964590056, -2.4851460551104267, -0.0519928693024332, -0.4862353313992518, 0.44456188897004295, 0.008265911865367051, -0.6717917956878144, -1.975908186905425, 0.7294526836258505, -0.7346205821636164, -1.7563161464696115, 0.34800412167464145, -2.0386550187205708, -0.6673662864341775, -2.1198491498654413, -0.3539027896294059, -1.75416108112623, 2.834487880207342, 1.0488722205740193, 0.39496029812332045, 0.27018732194101197, 0.5776679417020493, 0.7703154591412372, 0.19582266942496684, -1.1720472389774204, 0.0846491556289157, -1.6462053705124124, -0.44313175292018014, 0.323804186990013, -1.1715265703074953, 0.3597177162640596, -0.5640355938346974, 2.305788817458767, -0.45875279541128033, 0.4194607028043974, 0.5428640977973445, 0.06497334286986749, 0.24855964241484493, -0.3369352511490387, 0.0680285031317718, 0.1134094815639221, 0.768278430806495, 1.2186624025910853, -0.0597823714403905, 1.3771777788880015, -0.03466918364848714, -0.9716220453938154, -0.45601677708758465, -0.680123416361509, -0.23667272619480517, 0.057438340026504546, 0.2153534625582482, 1.1716027851332038, 0.7437801546212626, -0.42823844985611775, 0.5515534737636465, -0.06826337875483947, 0.7752611319075144, 1.2428126618759907], [-0.11002772308008839, 0.771995067513167, 1.3816482171338023, 0.37883793803959076, -1.0862445667377492, -0.10389325347882274, -0.8879412748840606, 1.0032828019662987, 0.20885065248890933, 0.32705887314839616, 1.7815660940993387, 0.7549318243747614, 0.14609657646337676, -1.801280594650268, -0.5190239797800671, -0.6893594627343578, -0.18891308948146338, -1.2418675452121695, 0.6231746537438295, 1.2492834497798793, -0.35829185915167566, -1.46510614330511, 0.10252865048972155, -0.2410863930274411, 0.9897311588678164, -0.06414718306064189, -0.7486068103433263, -1.3904842097448644, 0.00418785330543149, -2.341910984263244, -1.1102516594402094, 0.7103474889491006, -0.16345555425083433, -0.05298767650404247, -0.2461647315508851, -1.3935243195851164, -0.3969093199015442, 0.4505880828440237, 0.5090497571521214, -1.0477505563543439, 0.06866946635068993, 0.9128543986485791, 0.2431277994500317, 1.5053487533088457, 0.0038011169394291775, -1.6429665644773435, -0.019247049945869347, 0.15031903739189023, 0.6627002295016187, 0.4961667802361744, 1.5101154353410713, 0.5983123958489449, 0.19232268687146328, -0.5222833046933711, -0.2363315416432069, 1.300779122033661, 1.8970719563057246, 0.3541885741468442, 0.2004421362064828, -0.1940967961496467, 0.24374691733254078, 0.06153979721202124, -0.4876436753253473, 0.9031162306916317, -1.0884913412824013, 0.1523299222087069, 0.16894540156143745, 0.6593641075725581, -0.1529067094190132, -0.9454064863868277, -0.09495012025156854, -1.022001326663921, -0.35295115961486445, -0.7605442050674466, 2.4143583693898027, -2.1618239337631975, -1.1992645554078847, 1.2234106120306143, -0.20800891432642177, -1.3707827980097076, 0.6209271756305005, 2.09519420926836, 0.032124526717779676, 0.8696923419259699, -1.3355820487857468, -0.5807186450435844, 0.2017858306843342, 0.1798940267670293, -0.6717209522905624, -0.5813283679593971, -0.616801699717753, -0.7839785684683802, -0.5484614892467495, -0.5689804769994137, 0.446039467060252, -1.4159428434426353, 0.49015129369678156, 1.739110753866204, 0.15974962922807984, -0.4689913869709015, -0.9315867724971494, -0.5179258400287515, -1.2062557584064257, 0.246588549739183, -0.10384680510048024, 0.7548412464105937, 0.26975369811948713, 0.8013891346642098, 1.511719804999348, 0.9802355575016852, 2.7696456760886554, 1.1745568451173047, -0.41266049615348, 0.6602440859430362, 0.81698332468367, 0.17481911419674082, 0.808191045922329, 0.6487068393691633, 1.4043909352288213, -0.03805595204103769, -0.8511533588796988, -0.3525236759403814, -1.846166872233831, -2.586589586222547, -0.988546397662261, -0.4793256572467963, -0.6566230985127698, 1.2159136706171545, -1.0604566529599728, -1.188367818595105, 2.0067546554657345, -1.5081034087597416, 0.7925783415034172, -0.08100622348978213, -2.5263479687581514, 1.0106369746643176, -0.3006312992435416, 0.40872962489624565, 0.2559392466566333, 0.3715325239735535, 0.937714054015126, 2.212381107137478, 0.30695539489357443, -0.1508513191886279, 0.9432128551857792, 0.5070323292661385, -2.4901311247357087, 0.4875846591430144, -0.35604800608352266, -0.20343689858683325, -1.4140553807006226, 1.9355500044737188, 0.5295946008822103, -0.42058772803078703, -0.3923244037349633, -0.1590702369155902, -0.08067242128696514, 0.8139740526615704, 0.41458773855855496, -0.4845865204732599, -0.21717733884443857, 0.6328940594669118, -0.9514304055494563, -0.38622396135031656, 0.9849122263597995, 0.029221597772020082, 1.470081093005204, -1.1047865214475814, -0.4134217820446244, -0.7660425749915964, -0.8202817899717333, 0.8061730116541087, 1.1120715404400714, 0.15595918062108516, -0.05185655965598293, 1.8509549913651506, 1.6825100778898776, -1.9854835987807657, -0.0036879734454724144, -1.9463562814166917, -1.464695116642166, -0.5834817566526397, -1.0735090794164543, 0.6736616073927093, -0.34390212045774143, -0.15754597145090576, -0.007964920723567004, 1.8153319117608178, 0.3829854603185151, 0.9934776673938464, 0.6037357036206406, 1.2935525401885035, 0.12952021536367253, 0.020334628108299897, 1.3615229582992723, 0.24919365145578698, -1.5153331399992136, -1.1802252839292504, -0.3561599985278383, 0.5753349816983261, -0.6934201079464714, 1.5747110615318618, 0.498045056497055, -0.1829458967991039, -1.140303188972525, 0.7791982068318923, 0.6921030916004454, -1.0838609066366927, 1.3912487119419543, 1.2913785700569964, 1.5427930056496155, 0.03739786889364022, 0.9968691393175174, -1.9821358160239835, 0.47722906826021094, -0.3579479805098296, -0.0430921239667274, 0.5005594088909417, -0.6542813403601672, -0.14153169376900354, -0.947636236843821, -0.7524847852175843, -1.086373176603313, 0.3781025108485659, 1.2850745441118652, -0.9349140322852776, -1.5641453540255577, -0.19115717824936715, 0.41853776155736033, 0.27368103161569957, -1.1483001696816955, 0.014793455630331164, -0.8309820472703866, -0.40998815644802983, -0.12410873867214033, -0.42377558703791185, 0.4590363403587066, -0.5017642683416035, 1.5069924427422559, 0.119850321176369, -1.119201555040754, 0.9781337885199607, 0.8552753470101706, 0.2674668685847507, 0.0023191686379985537, 0.3415566174648927, 0.8541526108295526, -0.12023637359215152, 0.8257984669911276, -0.560152053129629, 0.49979569043415645, -0.9818474738046098, 0.3501665049445123, 0.5734096105622517, -0.5280386826020455, 0.3668795022209412], [-1.835929315871268, -0.09078956931043292, -0.2519560918319871, 0.6261123820015617, -0.03216978601132502, 1.0157328080790997, 1.8918680357827395, 0.10988521879019511, 0.8809043511026878, 1.66884062450296, 0.9587736731325951, 1.1014141628782204, -0.1810910213377079, 0.41558181297782765, -0.9561450057881312, 0.8962194851694809, 0.4647762458586043, 0.4695831969229873, -1.7807432184918313, 1.063197133129998, 1.5631680542781046, 1.4372722782154084, 0.520931868353026, 1.3847224020488236, 0.11799574853489692, -0.555206807948779, 0.036239781216835444, -0.6203915254748261, 1.8064748876044803, -2.3285765415437893, 0.321563458117661, -0.7582954048387482, -0.4648258720643729, 0.7721019511243973, 0.0010728732946601927, 1.1398021714886633, -0.07550903620058083, -0.0626789594285195, 0.1525497151073707, 1.2710642621898691, -0.6191658569604462, -0.2611549416646746, -0.4995293200990924, -0.41249682038417196, -1.1698218399452138, -0.27102804087059973, 0.45602171523606194, 1.4028850524220373, 0.47913348907786735, -0.5484724264174405, 1.4301501268446362, -1.6629748825063184, 0.4938994538534128, -1.1648668165284919, -0.11364044202414084, -1.29550596865973, -1.4738614973709814, -0.5260666102886647, 1.7430178616841019, 0.3578635398546071, 0.888338895797999, -0.3118457523893193, 0.0351702303015058, -0.7313893195793518, -0.5544531135621488, -0.3021631023515722, 0.9418635195389845, 0.3354870638096447, -0.6983244879896803, -0.14894659270620295, -0.8271365638380996, 0.5408108851194846, 0.2781442809635722, 0.2684118018364643, 0.7127909404460796, 0.5760357126398209, 1.451808160862495, -0.6904739265648905, 0.7565162019236474, -0.12947697099114638, 0.9450423423179922, -0.4438964312228485, -1.5570639095408196, -0.36348033554312215, -0.5647035592226829, -1.0048436287324118, -0.31766147599215433, 1.165823581697923, -0.2769258260354019, 0.7103110423406386, 1.382647757548218, 1.048592667727349, 0.5053790408273128, 0.9470524099027038, -0.36324417147950017, -1.2114106745994113, -1.4156873368929666, 0.9537521436558866, 1.3702075300487313, -0.24092398432795628, 0.07372089700690877, -2.195412179120931, 0.7493799812129122, -0.05921966383329219, -1.7598094183590207, 0.5254387732815018, 0.628457700777155, 0.8586252336515959, -0.061409397865717454, 0.2952809627386788, -0.7983996834714429, 1.4147004170191924, -0.2739915777903609, 1.2805297497865264, 0.5807191718878045, -0.16055175159425475, 1.3381667750297863, -0.5708510389840105, 0.4306040141589213, 0.2118099309721837, -0.05073472294093635, 0.2609821108419961, 0.7672942647696418, -0.2867828680340926, -0.4026758467524258, -0.1136144283256641, -0.458624695151, -0.8841323011228207, 0.3353229548601589, -1.622040205804831, -0.037566306888309725, -0.42179671814047676, 0.17866653126690524, 0.46922632470579906, 0.25976164555459275, 0.0031017425303257566, 1.2936143477055317, -0.49640031406723606, -1.1512334172810534, -0.22101573665658028, 0.6901533169803908, 1.76496634485979, 2.047253933735794, 0.38911056635874275, 1.162816785482709, 1.617887256042534, -0.4164877167388973, 1.502710250397837, 0.4893494956306787, -0.05473913754012153, 0.5963488254812577, 0.7002539423248872, -1.8846111395179395, 0.13665809219400443, 0.26840599860593534, -0.6256026944054554, 0.22466111726602392, -0.33544455043032484, -0.7364911776849765, -0.07741631618765155, -1.8945853393086094, 0.4436979150878056, 0.5508512453721855, 0.3339823630364551, 0.5648524813896552, -0.39473009776230966, -0.526307391523521, -1.687356810159487, -1.666625537363544, -0.71474446269074, 0.9024737661826285, -0.013195346192148079, 1.3097249781404952, 0.7569389782506931, -0.9791737578341988, -0.566962944458409, -1.0532512905001739, -1.4057398357597373, 0.24755068052369592, 0.6550900634331187, -0.8716891495235984, 1.034915284967327, 0.8657375930348863, -0.5404920262009648, 0.3483237753037023, -0.9206127476127705, 0.036037797183255824, 0.5312561272495329, -1.7815131940344984, 1.4506588026442504, -1.6302108060504172, 0.11421105156015214, -0.4441267016082362, 0.8888137617026355, -0.8200782127560787, -0.4398394023073438, 3.0837292139859924, 1.7044959589627973, 0.8938468473554116, -0.6433340440396043, 0.3887316850378692, -1.5694147013667845, -0.7281180267875325, 0.4661628508962295, 0.8156569182741148, -0.2960285781189564, 0.7582726960293572, 0.860257206658062, -0.9247071442324957, 0.2795711148469217, -0.012307039662657596, 1.4392147277477016, -1.1857545889916152, 0.4751496851366976, -0.12353600296177988, 2.028832127969138, -2.221548232360594, 1.7025282484150568, -0.15964986436698977, 0.5537568145197037, -1.4458271586751443, -1.2663452103879829, -0.22937640437974968, 0.09224818173262457, -2.39435068009697, -0.6377476877837828, -0.44123694590902157, -0.1522292943436161, 0.6725818600434305, 0.311725345223622, -0.25436833312453905, -1.2076349331480642, -1.5917502958626648, 0.09833915634133272, 1.4458619139178854, 0.013866340588021615, 0.9175162453259622, -1.1154683169933255, 0.9184733601959689, -0.19478222876728984, 0.11630292070420413, 0.6666701588425985, 0.7654075128093978, 0.2749263057094867, 0.39871650272791903, 0.3405900790192682, -0.6846163808407346, -0.2864117606840036, 0.027511447991102354, 1.6294236355826843, -1.1528058652928994, 0.11246739192555291, 1.0006275043925443, 0.6943629431246331, -0.8954714185650386, -1.223418350328228]], "dtype": "float64"}, {"__nd__": [[-0.5150039648661323, 0.31044102953349517], [0.8153358685028201, -0.19246058832427118], [0.13214771265192402, -0.37507049539712783], [-0.8055791784133176, 1.2113840458463423], [1.738130794879934, -0.13199988076413244], [-0.7902155893993617, 0.6571114117272728], [-1.4310196822570724, -1.2993635235727978], [0.15880897770189284, 1.805622373816603], [-0.3640054363397919, 0.07365165658622101], [1.5364117611330828, -1.1447249161612136], [-1.5194869686166184, 2.1475193038843643], [1.0132942831193228, -0.37201597440721873], [0.5280509060804656, 0.9129680351718164], [-0.24364336525545957, 2.505034131568372], [-1.600068229318896, 1.609355641388779], [0.40544281543071636, 0.20331437506117972], [0.2786067634860276, -0.6099189900556883], [-0.17091175360484745, -0.46974820374670817], [-1.4039156962112318, 0.6048540836190934], [-1.2970548507038822, -0.9390992483645973], [1.934893801187398, -0.49795796114059054], [1.5309874499207632, 1.3186101439847366], [0.223057280630069, 1.552408688463736], [-0.5529058071758425, -0.870555318905357], [-0.45845603353540443, -0.3535566341232156], [0.49214005385837206, 1.941741939435447], [-0.8334349089438704, -0.27879322860018163], [-0.029568483834502408, 0.25697549928106334], [-0.3292519942113516, 1.3516949228323172], [-0.39768415999081114, -0.9375371879868879], [1.6618131805512633, 1.3922444394442117], [-0.01570699175557951, -0.8827831189880472], [0.1610074968952387, 0.7203803244803844], [1.5610097576639428, 1.6520632579708734], [-0.26597887344120685, 0.10023519315738362], [-0.1683473241547368, 0.0845368903260268], [-0.4234417460558838, -0.333516692293782], [-0.12976913927510822, 0.1400381049133549], [-1.7735681660699223, -0.370677315215562], [0.7609299996925418, 0.4752819825009776], [0.7215958250582493, 0.7713306836471754], [0.564183974815629, 1.0729671089808517], [0.06073113207508135, -1.176886856513595], [1.275486041769026, 0.38339112575054324], [-1.66884844864153, 0.8746173547311442], [0.8135969923958581, -1.2649506637020358], [0.4574744043829685, -0.016305931992910885], [0.38968631289716377, -0.5415145854777887], [1.6936803855514484, -0.9156949270595771], [-1.023964805102812, -0.4161851102460136], [-0.03805579772534784, -0.560037379130133], [1.4762362258238926, -1.0005455638891725], [2.3036196083930016, -1.4103734977019633], [0.45195498122088834, 1.7384062913792535], [2.3185143056118878, -0.19672796398111927], [-1.4557608803899489, -0.6845086734254313], [-0.4879689662070506, 0.6899071209133352], [-0.9203314250013507, 0.9694202046985991], [-0.8887831734768346, 0.836791636094157], [2.5651392148848897, 0.802171666502174], [1.2293582475038551, 0.2604033953937031], [-0.22875114291066914, 0.601517195035194], [0.6870863658158144, -2.7567304874306404], [-0.7065978740289947, 2.163733407253192], [0.6743221023320213, 0.2892206389814026], [-0.2999726672110889, -0.2888916072713896], [-0.06420105494922561, -3.13185736920065], [1.7845333767605869, 0.4591151087432597], [-0.719586927334359, 1.3803193463252033], [-0.06316171745753461, 1.4616949247559643], [-0.5566780166879368, 0.8819375127625221], [-1.5893584003903733, 0.9194385816680429], [-1.4378962297937323, 0.0712535667857949], [0.46319515192579375, -1.5581746988946017], [0.19121149155123854, -0.39268137338477366], [0.17770609455126615, 1.0356705754868574], [0.06680100447100844, 1.3153940595941438], [-0.7574915682749487, -0.8923066341426112], [1.0076894237488268, -0.9810171801596419], [0.41732743082432844, -0.7940329870112848], [-0.011571998715065309, 0.4518110118728333], [-0.6925457762382944, -0.6430832937770593], [-1.3995842422576332, 0.8500143919276965], [-0.8561331061948305, -0.9260458159773721], [-0.05563884114670976, -0.3447218585491927], [-0.8094437256352941, 0.3255534885395341], [0.0886555112253823, -0.6936453211300572], [-0.5070746346621882, 1.2352531274554945], [0.07822833635652447, -0.5477584063359086], [-0.13138717424052063, 0.5069939300094514], [1.8589745922801337, -1.2020351722975404], [0.39697276568403655, -0.579070662703932], [0.2755287954016405, -0.5759736772616153], [-0.749897241394276, -0.7704034640667595], [-0.5163504929709988, 0.2575439844403928], [1.8688461228023443, -1.017458513928004], [-0.8248863219279488, -0.4684139969961896], [1.5056358373666778, 1.4552286955437164], [1.4039926474506226, 0.5369095048466942], [0.0044518844231870814, -1.1720294727220704], [0.07408120034884381, 0.31042571314142176], [-0.43988744792054735, -0.6279899637757163], [-1.1705458025682995, 0.41006034357813737], [-0.12239393521417362, 0.9052042759396429], [-0.6321046463871857, -0.5911554320460547], [1.163968671452811, 0.0827516979863224], [-1.5425097032148845, -0.7358383694550373], [-1.0163053048364061, -0.20133687184711677], [-1.887832915115647, 0.6521265636662636], [0.6460658059155451, -0.11271404385862976], [-0.10191445712621505, -0.10257471919384793], [-0.5046232754197436, 1.0664208499727292], [-1.0033553776189272, 2.3478959678084155], [-1.070341510633924, 0.2739188510461468], [-0.9623493310858114, -0.7673089346364427], [-0.17637316751266202, -1.498134134135622], [0.531027097040577, 0.18474519075315393], [0.549400642518245, 0.7451784405753021], [-0.7986694859785882, -0.18345377810221777], [-0.2241459123893825, 1.117468235138092], [-0.11196750184297002, -0.8403891251045706], [0.3524259669976004, -0.4039482446939817], [0.8629251118522256, 1.3661443883692765], [0.9894413255284961, -0.7569129692974765], [0.8981387256863476, -0.8568196905244344], [-0.34736821998343986, 0.9654113551106216], [0.23857157059203798, 1.8532526345364446], [0.8401405069319983, 0.21019379351998285], [1.1420434650316562, 0.10905522731713908], [1.312745670443575, -2.1813840741448245], [-0.08332618198595679, -0.9106487960738306], [-0.14007168860362737, -0.5657738796437249], [1.9221839443381747, 0.5714936909843473], [-0.10705722481893581, 0.870194732282334], [-2.469537323056538, 1.342334502062032], [-2.0629871150171413, 0.1256365003630935], [1.9234081053584358, -1.1558978671524056], [-0.9116812072346813, -2.2435436482641795], [-1.8707718024321804, 0.24214289603641986], [0.14352145546500217, -0.4831598126235153], [0.3368981923006493, -0.47746267159095884], [-0.9098635216597987, -0.4196004902198744], [-1.202274663363264, -2.3262914317894428], [-0.2732092661818778, 0.39259895221665125], [1.165679266896576, 0.618947835897744], [-0.30879239678034065, -0.8440166065130087], [-0.10659139221162563, 0.31983262511617316], [-0.06477137652025192, -0.5099008374168104], [0.37618313099362427, -2.2963836930791808], [0.6410806206417052, 1.7103403480173673], [-1.2511530771021406, 0.6083865807678422], [-0.3924522602611801, -0.7482646726970126], [0.6806286494705001, 0.6352661176224258], [-0.16396353628670177, 1.7646688167949727], [0.5475361703334843, -1.7912185896950668], [1.5557266682601156, 0.14678419382361899], [1.6264749111692178, -0.9556404385775934], [-0.8795748483128025, -0.42757999180776834], [0.3617802092118036, 0.7557139794027574], [-0.7181958414319247, 3.462194029469756], [-0.9120350991807551, -0.3878888909910339], [2.0478523932215063, 1.9097438031792067], [1.5447864765190278, 0.40868745194995626], [0.1553781530362207, 2.273048874348231], [-1.2491005874915746, -0.5726447362673527], [0.05652293452233668, 0.21013888529606656], [1.069345502851216, 0.12016403637085031], [0.31630029224687345, 0.4735610476410175], [-0.29353300551131845, 0.24181777255389852], [-1.1546590671260586, -0.9195194982178264], [0.006813918869633131, -0.3537650450825356], [0.611691684346923, 0.03901644065512309], [0.26846724170420116, 0.9498339073417772], [-1.7391348337303523, -1.1106218251920505], [-0.3248790825442782, 1.7927188050985787], [-0.429773355783865, 1.7889757376043267], [1.0059052006710125, 0.11098102217318238], [0.41506219381999715, 0.5857453343053213], [-0.4210322729309401, 1.469935755156165], [0.45957354679754, 0.008044858281748541], [-0.5922906426908746, 0.5752798260189618], [0.5780690281034969, 0.8597188400368174], [2.2893491142208844, 0.654285066841307], [-0.5923194132313303, -0.6310979379679095], [-1.2207721871296133, -0.9401594895513992], [-1.2583428288487812, 0.9791866483454279], [-2.1753428899753224, 1.4467062529381218], [-0.6013874095003379, -1.3323946818225771], [-0.8040574045636723, -0.04079132415209303], [0.06494937259726533, -0.4869627225446303], [1.050710829298522, -1.2806959198309187], [-0.915168248835826, 0.7193975616357081], [-1.2816697448555667, -0.7660837631445213], [0.5429093038694214, 0.6159173428524236], [0.5999884311427305, 0.6730370628239022], [0.0025066073660937964, 0.2224608905813923], [0.16672032501294545, -0.030960195510332855], [0.9549647343800025, 0.05329661387090327], [0.8897465835147325, 0.04261633730905357], [0.9419011649974716, -2.2773066303053016], [2.039129394038364, -0.828705427480821], [-0.29510884797672554, -0.44015932144621867], [1.9259736607406066, -1.3869355326592832], [-0.2882935940578264, -0.2197601546307481], [2.39188272812458, 1.4288609850957772], [-0.8298737372907121, -0.2317103272169055], [0.17513692403678918, -0.7443200596796999], [1.4674370774607173, 0.3559743478880168], [-1.6432822452530913, 0.10888207416085754], [-0.02062751378639602, -0.44521064593999643], [0.2871344574689811, 0.7368882458839546], [-0.22309185730275413, -0.5899825567817608], [0.3893317700436249, -1.694982737183782], [1.0179726964390077, -1.297396114794926], [0.17168649833096977, -0.15466695769447134], [0.5610171979167221, -0.5463687856944385], [0.42716801656738224, -1.6153374344095512], [1.4065680738079376, 0.5812361999913319], [0.968129472343223, -0.49858469379049025], [0.2622463395792193, -0.39078217115658415], [-0.00144827741078281, -1.0629672144638225], [-0.0537701341007373, -2.5877890577636506], [1.1154364136608108, -1.2140303742764775], [0.8920956670636366, 1.3955110300896962], [0.7153454161236134, -1.1201954941308483], [0.3461054223388797, 0.7404934881405467], [0.8061955972697551, 0.44843903426049736], [0.10814400325054929, -0.28478937347235417], [-0.23548944242488068, 0.1981731839458969], [-0.048447953909042815, -0.27674804468591496], [-0.33043239833456384, 0.0799173741411519], [-0.5527611597959253, 0.9548526134368582], [0.08763328133627685, 1.2040511050007447], [-0.6846524125151624, -0.8584910815471601], [-0.5123648069448085, -0.08823669971064897], [0.02501223651059791, 1.0101319733130791], [-2.0360263616729792, 0.2747221244062303], [-1.6047976734431904, -0.9040473513895094], [-0.16172225990693267, 0.07164842938842603], [1.3401494731369752, -0.198916084271783], [1.2449789122473078, 0.6525619923854502], [-0.16122118341313785, 0.9540615213455168], [-0.4087699521155149, 0.5934054281548816], [1.9269952655445166, 0.7427927782602494], [-0.014197665325094833, -0.6205947093440051], [-0.24017492369477733, -0.5838214226903168], [1.8216199489012368, 0.8888182359290911], [0.21033838253967657, 0.513004848132024], [2.0210383866515684, -0.1579497572619202], [-0.25414464634188155, 1.6138279679519725], [0.8028695057970964, -0.9413954439862449], [0.7036400987528975, 0.06840905376111647], [-0.15852188773170003, -0.2885330370982452], [-0.390647362771925, 0.23674175421492502], [0.9215209114080298, -0.842981970487534], [0.5361756825057775, -1.4044854226288794]], "dtype": "float64"}], "e": {"__tup__": [{"__nd__": [[-7.030338764190674, 5.122753620147705], [-28.142986297607422, 20.01205825805664], [-31.546844482421875, -10.348849296569824], [24.219057083129883, 13.382488250732422]], "dtype": "float32"}, {"__nd__": [[-16091, 8687], [-47927, 25250], [-71619, -17407], [49383, 20217]], "dtype": "int32"}, {"__nd__": [[-25, 11], [40, -7], [7, -14], [-40, 44], [86, -5], [-39, 24], [-71, -48], [8, 66], [-18, 3], [76, -42], [-75, 79], [50, -14], [26, 33], [-12, 92], [-79, 59], [20, 7], [14, -22], [-8, -17], [-70, 22], [-64, -34], [96, -18], [76, 48], [11, 57], [-27, -32], [-23, -13], [24, 71], [-41, -10], [-1, 9], [-16, 50], [-20, -34], [82, 51], [-1, -32], [8, 26], [77, 61], [-13, 4], [-8, 3], [-21, -12], [-6, 5], [-88, -14], [38, 17], [36, 28], [28, 39], [3, -43], [63, 14], [-83, 32], [40, -46], [23, -1], [19, -20], [84, -34], [-51, -15], [-2, -21], [73, -37], [114, -52], [22, 64], [115, -7], [-72, -25], [-24, 25], [-46, 36], [-44, 31], [127, 29], [61, 10], [-11, 22], [34, -101], [-35, 79], [33, 11], [-15, -11], [-3, -115], [88, 17], [-36, 51], [-3, 54], [-28, 32], [-79, 34], [-71, 3], [23, -57], [9, -14], [9, 38], [3, 48], [-38, -33], [50, -36], [21, -29], [-1, 17], [-34, -24], [-69, 31], [-42, -34], [-3, -13], [-40, 12], [4, -25], [-25, 45], [4, -20], [-7, 19], [92, -44], [20, -21], [14, -21], [-37, -28], [-26, 9], [93, -37], [-41, -17], [75, 53], [70, 20], [0, -43], [4, 11], [-22, -23], [-58, 15], [-6, 33], [-31, -22], [58, 3], [-76, -27], [-50, -7], [-93, 24], [32, -4], [-5, -4], [-25, 39], [-50, 86], [-53, 10], [-48, -28], [-9, -55], [26, 7], [27, 27], [-40, -7], [-11, 41], [-6, -31], [17, -15], [43, 50], [49, -28], [44, -31], [-17, 35], [12, 68], [42, 8], [57, 4], [65, -80], [-4, -33], [-7, -21], [95, 21], [-5, 32], [-122, 49], [-102, 5], [95, -42], [-45, -82], [-93, 9], [7, -18], [17, -18], [-45, -15], [-60, -85], [-14, 14], [58, 23], [-15, -31], [-5, 12], [-3, -19], [19, -84], [32, 63], [-62, 22], [-19, -27], [34, 23], [-8, 65], [27, -66], [77, 5], [81, -35], [-44, -16], [18, 28], [-36, 127], [-45, -14], [101, 70], [76, 15], [8, 83], [-62, -21], [3, 8], [53, 4], [16, 17], [-15, 9], [-57, -34], [0, -13], [30, 1], [13, 35], [-86, -41], [-16, 66], [-21, 66], [50, 4], [21, 21], [-21, 54], [23, 0], [-29, 21], [29, 32], [113, 24], [-29, -23], [-60, -34], [-62, 36], [-108, 53], [-30, -49], [-40, -1], [3, -18], [52, -47], [-45, 26], [-63, -28], [27, 23], [30, 25], [0, 8], [8, -1], [47, 2], [44, 2], [47, -84], [101, -30], [-15, -16], [95, -51], [-14, -8], [118, 52], [-41, -8], [9, -27], [73, 13], [-81, 4], [-1, -16], [14, 27], [-11, -22], [19, -62], [50, -48], [9, -6], [28, -20], [21, -59], [70, 21], [48, -18], [13, -14], [0, -39], [-3, -95], [55, -45], [44, 51], [35, -41], [17, 27], [40, 16], [5, -10], [-12, 7], [-2, -10], [-16, 3], [-27, 35], [4, 44], [-34, -31], [-25, -3], [1, 37], [-101, 10], [-79, -33], [-8, 3], [66, -7], [62, 24], [-8, 35], [-20, 22], [95, 27], [-1, -23], [-12, -21], [90, 33], [10, 19], [100, -6], [-13, 59], [40, -35], [35, 3], [-8, -11], [-19, 9], [46, -31], [27, -52]], "dtype": "int8"}, {"__nd__": [0.020197946578264236, 0.027261368930339813], "dtype": "float32"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(int8_linear, json.loads(_TESTS), 'jax', *[1e-05, 1e-06])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")